# Dipole Angle vs Parallactic / Zenith Angle Correlation

This notebook investigates whether the **dipole position angle** (`r:dipoleAngle`) and the
**dipole separation** (`r:dipoleLength`) measured by the Fink/LSST difference-imaging pipeline
are correlated with observing-geometry angles:

| Observable | Definition |
|------------|------------|
| **Parallactic angle** η | Angle between North and the great circle toward the zenith, at the object position |
| **Zenith angle** z | Angular distance from zenith to object (complement of altitude) |

A correlation with the parallactic angle would hint at an **atmospheric dispersion** or
**differential refraction** origin for the dipoles; a correlation with the zenith angle
would point to an **airmass-dependent PSF** effect.

## Angle-convention note

`r:dipoleAngle` from the Rubin AP pipeline is measured from the **+x pixel axis**
(pointing East in standard WCS), **counter-clockwise**.  
Astropy's `parallactic_angle()` follows the astronomical convention: **North = 0°, East = +90°**
(counter-clockwise when North is up).  
To compare the two quantities on the same footing we convert `r:dipoleAngle` to a
**position angle** (PA, North = 0°, East = +90°):

```
dipole_PA_deg = (90 - r:dipoleAngle) mod 360
```

All correlation analysis is done with `dipole_PA_deg`.  
The raw `r:dipoleAngle` is preserved for cross-checks and to stay consistent with notebook `01d`.

## Strategy

* Load dipole-only alerts from `data_DIPOLES_01c/` (same source as notebook `01d`).
* Compute η and z for each alert using `astropy` (RA, Dec, MJD → AltAz frame at Rubin).
* Study correlations **per DDF** and **per band**.
* Visualisations: scatter plots, 2-D histograms, polar plots, and Pearson/Spearman statistics.


- author : Sylvie Dagoret-Campagne
- affiliation : IJCLab/IN2P3/CNRS, Université Paris-Saclay
- creation : 2026-05-28
- last update : 2026-05-29 : add azimuth rose subplots by band per DDF (section 9b)
- last update : 2026-06-01

## 1. Imports & configuration

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
from scipy import stats

from astropy.time import Time
from astropy.coordinates import EarthLocation, SkyCoord, AltAz
import astropy.units as u

warnings.filterwarnings("ignore")
print(f"pandas   version : {pd.__version__}")
print(f"numpy    version : {np.__version__}")

In [ ]:
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl found → interactive backend (%matplotlib widget)")
except ImportError:
    %matplotlib inline
    print("ipympl NOT found → falling back to %matplotlib inline")

In [ ]:
# ── Input data (written by notebook 01c) ─────────────────────────────────────
DIR_DATA_IN = "data_DIPOLES_01c"

# ── Output figures ────────────────────────────────────────────────────────────
NB_TAG = "DIPOLES_05"
DIR_FIGS = f"figs_{NB_TAG}"
os.makedirs(DIR_FIGS, exist_ok=True)
print(f"Input data : {os.path.abspath(DIR_DATA_IN)}")
print(f"Figures    : {os.path.abspath(DIR_FIGS)}")

# ── Rubin/LSST observatory location (Cerro Pachón) ───────────────────────────
RUBIN_LAT_DEG = -30.244728  # degrees North
RUBIN_LON_DEG = -70.749417  # degrees East  (West is negative)
RUBIN_HEIGHT_M = 2647.0  # metres above sea level

RUBIN_LOCATION = EarthLocation(
    lat=RUBIN_LAT_DEG * u.deg,
    lon=RUBIN_LON_DEG * u.deg,
    height=RUBIN_HEIGHT_M * u.m,
)
print(f"Rubin/LSST : lat={RUBIN_LAT_DEG}°  lon={RUBIN_LON_DEG}°  h={RUBIN_HEIGHT_M} m")

# ── LSST Deep Drilling Fields ─────────────────────────────────────────────────
DEEP_FIELDS = {
    "COSMOS": (150.1191, 2.2058),
    "ELAIS-S1": (9.4500, -44.000),
    "ECDFS": (53.1250, -27.800),
    "EDFS-a": (58.9000, -49.315),
    "EDFS-b": (63.6000, -47.600),
    "EDFS": (61.2400, -48.423),
    "M49": (187.4000, 8.000),
}

# ── Plotting style ────────────────────────────────────────────────────────────
BAND_COLORS = {
    "u": "#9b59b6",
    "g": "#2ecc71",
    "r": "#e74c3c",
    "i": "#e67e22",
    "z": "#3498db",
    "y": "#795548",
}
BAND_ORDER = list("ugrizy")

plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 9,
    }
)


def savefig(name: str) -> None:
    """Save current figure as PDF and PNG in DIR_FIGS."""
    for ext in ("pdf", "png"):
        plt.savefig(os.path.join(DIR_FIGS, f"{name}.{ext}"), bbox_inches="tight")
    print(f"  -> saved {name}.{{pdf,png}}")


print("Configuration done.")

## 2. Angle-convention helpers

### Why the conversion is necessary

| Angle | Origin axis | Direction |
|-------|-------------|----------|
| `r:dipoleAngle` (Rubin AP) | **+x pixel** = East | counter-clockwise |
| Astropy `parallactic_angle()` | **North** | counter-clockwise (East = +90°) |

The two are related by a simple 90° rotation:
```
dipole_PA = (90 - r:dipoleAngle) mod 360
```

Note: `r:dipoleAngle` describes a **headless** axis (0° and 180° are the same orientation),
so when comparing Δ = `dipole_PA − parallactic_angle` we also look at the
**folded** version Δ_fold = min(|Δ|, 360−|Δ|) ∈ [0°, 180°].

In [ ]:
def dipole_raw_to_PA(dipole_angle_deg: np.ndarray) -> np.ndarray:
    """
    Convert Rubin AP dipoleAngle (measured CCW from +x/East pixel axis)
    to standard astronomical Position Angle (measured CCW from North).

    PA = (90 - dipoleAngle) mod 360

    Parameters
    ----------
    dipole_angle_deg : array-like, degrees

    Returns
    -------
    pa_deg : ndarray, degrees in [0, 360)
    """
    return (90.0 - np.asarray(dipole_angle_deg, dtype=float)) % 360.0


def angular_difference_headless(a_deg: np.ndarray, b_deg: np.ndarray) -> np.ndarray:
    """
    Compute the signed angular difference (a - b) modulo 360,
    then fold to [0, 180] to account for the headless nature of dipole axes.

    Parameters
    ----------
    a_deg, b_deg : array-like, degrees

    Returns
    -------
    delta_deg : ndarray in [0, 180]
    """
    diff = (np.asarray(a_deg, dtype=float) - np.asarray(b_deg, dtype=float)) % 360.0
    return np.where(diff <= 180.0, diff, 360.0 - diff)


# ── Quick sanity check ────────────────────────────────────────────────────────
test_raw = np.array([0.0, 90.0, 180.0, 270.0, 45.0])
test_PA = dipole_raw_to_PA(test_raw)
print("Convention check:")
print("  dipoleAngle (raw) :", test_raw)
print("  dipole_PA         :", test_PA)
print("  Expected          :  90  0  270  180  45 (mod 360)")

## 3. Observing-geometry helper functions

We compute, for each alert:
* **Parallactic angle** η — given directly by `AltAz.parallactic_angle` in astropy.
  Convention: North = 0°, East = +90° (counter-clockwise).
* **Zenith angle** z = 90° − altitude.
* **Airmass** X ≈ 1/cos(z) (for cross-checks).

The parallactic angle and the hour angle are both computed from the LST:

$$H = \mathrm{LST} - \alpha \qquad\qquad
\eta = \arctan2\!\left(\sin H,\;\tan\phi\cos\delta - \sin\delta\cos H\right)$$

The function returns η, H (hours and degrees), azimuth, zenith angle, sin z, and airmass.

### 2a. `zenith_tangent_vector` — tangent-plane projection (from obstime)

Projects the zenith direction into the tangent plane of the target using
the formula $\mathbf{v} = \mathbf{z} - (\mathbf{z}\cdot\mathbf{s})\,\mathbf{s}$,
where $\mathbf{s}$ is the unit vector toward the source and $\mathbf{z}$ is the
zenith unit vector obtained by transforming AltAz(alt=90°) to ICRS.
This version calls `astropy` for the time transform and is used for individual
sanity checks.

In [ ]:
def zenith_tangent_vector(ra_deg, dec_deg, obstime, location):
    """
    Compute the projection of the zenith direction into the plane tangeant to the object
    using the formula  :
                      v = z - np.dot(z, s) * s
    where s is the direction of the source, and z the direction of zenith

    Parameters:
    ==========
        ra_deg,dec_deg: target coordinates in the sky
        obstimes:  observation times
        location: localtion of observatory

    Returns:
    =========
        array of unit vectors in the tangeant plane
    """

    # Source
    sky = SkyCoord(ra=ra_deg * u.deg, dec=dec_deg * u.deg)

    # Zénith en AltAz → (alt=90°, az arbitraire)
    zenith_altaz = SkyCoord(alt=90 * u.deg, az=0 * u.deg, frame=AltAz(obstime=obstime, location=location))

    # Convertir en ICRS
    zenith_icrs = zenith_altaz.transform_to("icrs")

    # Vecteurs cartésiens
    s = sky.cartesian.xyz.value
    z = zenith_icrs.cartesian.xyz.value

    # Projection tangentielle
    v = z - np.dot(z, s) * s

    # Normalisation
    v /= np.linalg.norm(v)

    return v  # vecteur 3D tangent au ciel

### 2b. `zenith_tangent_vector_fromHA` — tangent-plane projection (from HA grid)

Same projection as above but computed analytically from a **precomputed hour-angle array** —
avoids repeated `astropy` time calls and is fast enough to sweep the full
$H\in[-180°,+180°]$ range for all DDFs.
The function also returns $\|\mathbf{v}\| = \sin z$, the zenith-angle sine
that governs DCR amplitude.

In [ ]:
def zenith_tangent_vector_fromHA(HA_deg, coords, location):
    """
    Compute the projection of the zenith direction into the plane tangeant to the object
    using the formula  :
                      v = z - np.dot(z, s) * s
    where s is the direction of the source, and z the direction of zenith
    Parameters:
    ==========
        HA_deg : array of Hour angles
        coords: target SkyCoords
        location: localtion of observatory

    Returns:
    =========
        array of unit vectors in the tangeant plane
        array if sinz values (related to dipole intensity)
    """

    lat_deg = location.lat.to(u.deg).value

    dec_deg = coords.dec.to(u.deg).value
    ra_deg = coords.ra.to(u.deg).value

    ra = np.deg2rad(ra_deg)
    dec = np.deg2rad(dec_deg)

    # --- direction source ---
    s = np.array([np.cos(dec) * np.cos(ra), np.cos(dec) * np.sin(ra), np.sin(dec)])  # (3,)

    # --- zénith ---
    HA_val = HA_deg.to(u.deg).value  # ← FIX unités
    lst = np.deg2rad(HA_val + ra_deg)
    lat = np.deg2rad(lat_deg)

    z = np.array(
        [np.cos(lat) * np.cos(lst), np.cos(lat) * np.sin(lst), np.sin(lat) * np.ones_like(lst)]
    )  # (3, N)

    # --- projection ---
    # v = z - np.dot(z, s) * s
    proj = np.sum(z * s[:, None], axis=0)  # (N,)
    v = z - proj * s[:, None]  # (3, N)

    # --- norme par point ---
    norm = np.linalg.norm(v, axis=0)  # (N,)

    # --- normalisation optionnelle ---
    v_unit = np.zeros_like(v)
    mask = norm > 0
    v_unit[:, mask] = v[:, mask] / norm[mask]

    return v_unit, norm

### 2c. `sinz_vs_HA` — zenith-angle sine from analytic formula

Direct analytic computation:
$$\sin z(H,\delta,\phi) = \sqrt{1 - \left(\sin\phi\,\sin\delta + \cos\phi\,\cos\delta\,\cos H\right)^2}$$
Used to cross-check the norm returned by `zenith_tangent_vector_fromHA`
and to overlay the alert's measured $\sin z$ on the model curve.

In [ ]:
def sinz_vs_HA(HA_deg, coords, location):
    """
    Compute the sinus of zenith angle from the formula
    \sin z(H,\delta,\phi) = \sqrt{ 1 - \left( \sin\phi\,\sin\delta + \cos\phi\,\cos\delta\,\cos H \right)^2

    Parameters:
    ==========
        coords: target SkyCoords
        times:  observation times
        location: localtion of observatory

    Returns:
    =========
        array of parallactic angles in degree

    """

    # --- location ---> latitude
    lat_deg = location.lat.to(u.deg).value

    # --- object ---> declination
    dec_deg = coords.dec.to(u.deg).value

    # --- HA be sure to have quanitites in deg
    HA_valdeg = HA_deg.to(u.deg).value

    HA = np.deg2rad(HA_valdeg)
    dec = np.deg2rad(dec_deg)
    lat = np.deg2rad(lat_deg)

    cosz = np.sin(lat) * np.sin(dec) + np.cos(lat) * np.cos(dec) * np.cos(HA)
    return np.sqrt(1 - cosz**2)

In [ ]:
def parangle(field, altaz_frame):
    """
    calculate the hour angle and parallactic angle  for a target given an AltAz frame.
    The one thing this routine does not do, is transform the parallactic angle back
    from FK5(equinox=obstime) to the frame of the input coordinates if wanted
    (SkyCoord provides easy transformation of coordinates between frames,
    but transforming position angles is not built in as far as I know).
    https://github.com/astropy/astroplan/issues/633
    """
    field_altaz = field.transform_to(altaz_frame)
    lon = field_altaz.location.lon
    lat = field_altaz.location.lat
    field_now = field_altaz.transform_to(FK5(equinox=field_altaz.obstime))
    ranow = field_now.ra
    decnow = field_now.dec
    lmst = field_altaz.obstime.sidereal_time("mean", longitude=lon)
    hourang = lmst - ranow
    parang = Angle(
        np.arctan2(np.sin(hourang), (np.tan(lat) * np.cos(decnow) - np.sin(decnow) * np.cos(hourang)))
    )
    return hourang, parang

### 2d. `calculate_parallactic_angle` — η from obstime

Computes the parallactic angle from first principles given `astropy` `Time`
objects:
$$H = \mathrm{LST} - \alpha, \qquad
\eta = \arctan2\!\left(\sin H,\;\tan\phi\,\cos\delta - \sin\delta\,\cos H\right)$$
Returns η in degrees, range $[-180°, +180°]$.

In [ ]:
def parallactic_angle_manual(ra, dec, lst, lat):
    # tout en radians
    H = lst - ra

    q = np.arctan2(np.sin(H), np.tan(lat) * np.cos(dec) - np.sin(dec) * np.cos(H))
    return q

In [ ]:
# -----------------------------
# My computation of  parallactic angle
# -----------------------------
def calculate_parallactic_angle(coords, times, location):
    """
    Parameters:
    ==========
        coords: target SkyCoords
        times:  observation times
        location: localtion of observatory

    Returns:
    =========
        array of parallactic angles in degree
    """

    # LST
    lst = times.sidereal_time("apparent", longitude=location.lon)

    # angle horaire H = LST - RA
    H = (lst - coords.ra).to(u.rad).value

    # latitude
    phi = location.lat.to(u.rad).value

    # déclinaison
    dec_rad = coords.dec.to(u.rad).value

    # formule du parallactic angle
    sinH = np.sin(H)
    cosH = np.cos(H)

    tan_phi = np.tan(phi)

    num = sinH
    den = tan_phi * np.cos(dec_rad) - np.sin(dec_rad) * cosH

    q = np.arctan2(num, den)

    return np.degrees(q)

### 2e. `calculate_parallactic_angle_fromHA` — η from HA grid

Same formula as above but accepts a **precomputed hour-angle** `Angle` object
(in degrees) instead of `Time`.  Used to sweep the full HA range for
all DDFs without triggering IERS/UT1 network calls.

In [ ]:
# -----------------------------
# My computation of  parallactic angle
# -----------------------------
def calculate_parallactic_angle_fromHA(ha, coords, location):
    """
    Parameters:
    ==========
        coords: target SkyCoords
        ha:  hour angle Angle in degree
        location: localtion of observatory

    Returns:
    =========
        array of parallactic angles in degree
    """

    # angle horaire H = LST - RA
    ha_rad = ha.to(u.rad).value

    # latitude
    phi = location.lat.to(u.rad).value

    # déclinaison
    dec_rad = coords.dec.to(u.rad).value

    # formule du parallactic angle
    sinH = np.sin(ha_rad)
    cosH = np.cos(ha_rad)

    tan_phi = np.tan(phi)

    num = sinH
    den = tan_phi * np.cos(dec_rad) - np.sin(dec_rad) * cosH

    q = np.arctan2(num, den)

    return np.degrees(q)

In [ ]:
def compute_parallactic_zenith(
    ra_deg: np.ndarray,
    dec_deg: np.ndarray,
    mjd: np.ndarray,
    location: EarthLocation = RUBIN_LOCATION,
    batch_size: int = 500,
) -> pd.DataFrame:
    """
    Compute the parallactic angle and zenith angle for a set of sky positions
    observed at given MJD times from a given ground location.

    Parameters
    ----------
    ra_deg, dec_deg : array-like ICRS coordinates in degrees.
        Observed coordinates array in ra and dec
    mjd : array-like
        Observation times in MJD (TAI).
    location : EarthLocation
        Observer position on Earth.
    batch_size : int
        Number of alerts processed per astropy call (trade-off speed vs memory).

    Returns
    -------
    pd.DataFrame with columns:
        parallactic_angle_deg : float  (−180 … +180 deg)
        zenith_angle_deg      : float  (0 … 90 deg)
        altitude_deg          : float
        azimuth_deg           : float
        airmass               : float  (≈ 1/cos(z))
    """
    ra = np.asarray(ra_deg, dtype=float)
    dec = np.asarray(dec_deg, dtype=float)
    t = np.asarray(mjd, dtype=float)
    n = len(ra)

    para = np.full(n, np.nan)
    q_altaz = np.full(n, np.nan)
    za = np.full(n, np.nan)
    alt = np.full(n, np.nan)
    az = np.full(n, np.nan)

    for i0 in range(0, n, batch_size):
        sl = slice(i0, min(i0 + batch_size, n))
        try:
            # le temps sidéral est défini en UT1, pas en TAI
            # times = Time(t[sl], format="mjd", scale="tai")
            times = Time(t[sl], format="mjd", scale="tai").ut1

            # target coordinates
            coords = SkyCoord(ra=ra[sl] * u.deg, dec=dec[sl] * u.deg)
            ra_rad = coords.ra.to(u.rad).value
            dec_rad = coords.dec.to(u.rad).value

            # LST locale
            lst = times.sidereal_time("apparent", longitude=location.lon)
            # angle horaire
            # H = (lst - coords.ra).to(u.rad).value
            H = (lst - coords.ra).wrap_at(180 * u.deg).to(u.rad).value

            # create the Rubin Observatory frame
            frame = AltAz(obstime=times, location=location)
            altaz = coords.transform_to(frame)

            # calculate parallactic_angle does not exist altaz
            # para[sl] = altaz.parallactic_angle().to(u.deg).value
            # compute in degrees
            para[sl] = calculate_parallactic_angle(coords, times, location)

            # formule alternative via alt/az
            alt_rad = np.radians(altaz.alt.deg)
            az_rad = np.radians(altaz.az.deg)

            phi = location.lat.to(u.rad).value
            q_altaz[sl] = np.degrees(
                np.arctan2(np.sin(H), np.tan(phi) * np.cos(dec_rad) - np.sin(dec_rad) * np.cos(H))
            )

            alt[sl] = altaz.alt.deg
            az[sl] = altaz.az.deg
            za[sl] = 90.0 - altaz.alt.deg
        except Exception as exc:
            print(f"  [warning] batch {i0}–{i0 + batch_size}: {exc}")

    # Simple flat-Earth airmass (valid for za < 80°)
    with np.errstate(divide="ignore", invalid="ignore"):
        airmass = np.where(za < 89.0, 1.0 / np.cos(np.radians(za)), np.nan)

    return pd.DataFrame(
        {
            "parallactic_angle_deg": para,
            "q_altaz": q_altaz,
            "zenith_angle_deg": za,
            "altitude_deg": alt,
            "azimuth_deg": az,
            "airmass": airmass,
        }
    )


# Quick sanity check on a single alert
test = compute_parallactic_zenith(ra_deg=[150.1191], dec_deg=[2.2058], mjd=[60310.5])
print("Sanity check (COSMOS at MJD 60310.5):")
print(test.to_string(index=False))

In [ ]:
def compute_observing_geometry(
    ra_deg: np.ndarray,
    dec_deg: np.ndarray,
    mjd: np.ndarray,
    location: EarthLocation = RUBIN_LOCATION,
    batch_size: int = 500,
) -> pd.DataFrame:
    """
    Compute full observing geometry for a set of alerts.

    Parameters
    ----------
    ra_deg, dec_deg : array-like – ICRS coordinates in degrees
    mjd             : array-like – MJD TAI
    location        : EarthLocation
    batch_size      : int – alerts per astropy call (speed/memory trade-off)

    Returns
    -------
    pd.DataFrame with columns:
        parallactic_angle_deg  float   −180 … +180°   (North = 0, CCW)
        hour_angle_hr          float   −12 … +12 h     (H = LST − RA)
        hour_angle_deg         float   −180 … +180°    (same × 15)
        azimuth_deg            float      0 … 360°    (North = 0, E = 90)
        altitude_deg           float      0 …  90°
        zenith_angle_deg       float      0 …  90°
        sin_zenith             float      0 … 1
        airmass                float   ≥ 1             (≈ 1/cos z)
    """
    ra = np.asarray(ra_deg, dtype=float)
    dec = np.asarray(dec_deg, dtype=float)
    t = np.asarray(mjd, dtype=float)
    n = len(ra)

    para = np.full(n, np.nan)
    H_hr = np.full(n, np.nan)  # hour angle in hours
    az = np.full(n, np.nan)
    alt = np.full(n, np.nan)
    za = np.full(n, np.nan)

    for i0 in range(0, n, batch_size):
        sl = slice(i0, min(i0 + batch_size, n))
        try:
            # Sidereal time requires UT1
            times = Time(t[sl], format="mjd", scale="tai").ut1
            coords = SkyCoord(ra=ra[sl] * u.deg, dec=dec[sl] * u.deg)

            # LST and hour angle
            lst = times.sidereal_time("apparent", longitude=location.lon)
            H_wrap = (lst - coords.ra).wrap_at(180 * u.deg)  # Angle in (−180°, +180°]
            H_rad = H_wrap.to(u.rad).value
            H_hr[sl] = H_wrap.to(u.hourangle).value  # hours

            # Parallactic angle: η = arctan2(sin H, tan φ cos δ − sin δ cos H)
            phi = location.lat.to(u.rad).value
            dec_rad = coords.dec.to(u.rad).value
            para[sl] = np.degrees(
                np.arctan2(
                    np.sin(H_rad),
                    np.tan(phi) * np.cos(dec_rad) - np.sin(dec_rad) * np.cos(H_rad),
                )
            )

            # Alt/Az
            frame = AltAz(obstime=times, location=location)
            altaz = coords.transform_to(frame)
            alt[sl] = altaz.alt.deg
            az[sl] = altaz.az.deg
            za[sl] = 90.0 - altaz.alt.deg
        except Exception as exc:
            print(f"  [warning] batch {i0}–{i0 + batch_size}: {exc}")

    with np.errstate(divide="ignore", invalid="ignore"):
        airmass = np.where(za < 89.0, 1.0 / np.cos(np.radians(za)), np.nan)

    return pd.DataFrame(
        {
            "parallactic_angle_deg": para,
            "hour_angle_hr": H_hr,
            "hour_angle_deg": H_hr * 15.0,  # 1 h = 15°
            "azimuth_deg": az,
            "altitude_deg": alt,
            "zenith_angle_deg": za,
            "sin_zenith": np.sin(np.radians(za)),
            "airmass": airmass,
        }
    )


# Sanity check
test = compute_observing_geometry([150.1191], [2.2058], [60310.5])
print("Sanity check COSMOS MJD=60310.5:")
print(test.to_string(index=False))

## 4. Load dipole alerts from parquet files

In [ ]:
ddf_alerts: dict[str, pd.DataFrame] = {}

for field_name in DEEP_FIELDS:
    pq = os.path.join(DIR_DATA_IN, f"{field_name}_alerts.parquet")
    if not os.path.exists(pq):
        print(f"[{field_name:12s}] parquet not found — skipping.")
        ddf_alerts[field_name] = pd.DataFrame()
        continue

    df = pd.read_parquet(pq)

    # Cast boolean column
    if "r:isDipole" in df.columns:
        df["r:isDipole"] = (
            df["r:isDipole"]
            .map(
                lambda v: (
                    True
                    if str(v).strip().lower() in ("true", "1", "yes")
                    else False
                    if str(v).strip().lower() in ("false", "0", "no")
                    else pd.NA
                )
            )
            .astype("boolean")
        )

    # Cast numeric columns
    for col in (
        "r:midpointMjdTai",
        "r:ra",
        "r:dec",
        "r:dipoleAngle",
        "r:dipoleLength",
        "r:dipoleChi2",
        "r:dipoleFluxDiff",
    ):
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # Keep only dipole rows
    if "r:isDipole" in df.columns:
        df_dip = df[df["r:isDipole"].fillna(False).astype(bool)].copy()
    else:
        df_dip = pd.DataFrame()

    # ── Convert r:dipoleAngle → astronomical Position Angle ──────────────────
    # r:dipoleAngle is measured CCW from +x/East pixel axis.
    # Astronomical PA is measured CCW from North.
    # PA = (90 - r:dipoleAngle) mod 360
    if "r:dipoleAngle" in df_dip.columns:
        df_dip["dipole_PA_deg"] = dipole_raw_to_PA(df_dip["r:dipoleAngle"].values)

    df_dip["field"] = field_name
    ddf_alerts[field_name] = df_dip

    print(f"[{field_name:12s}] {len(df):7,} total alerts  |  {len(df_dip):6,} dipoles")

print("\nLoad complete.")

## 5. Compute parallactic & zenith angles for every dipole alert

In [ ]:
frames_with_angles: list[pd.DataFrame] = []

for field_name, df_dip in ddf_alerts.items():
    if df_dip.empty:
        print(f"[{field_name:12s}] no dipoles — skipping.")
        continue

    missing = [c for c in ("r:ra", "r:dec", "r:midpointMjdTai") if c not in df_dip.columns]
    if missing:
        print(f"[{field_name:12s}] missing columns {missing} — skipping.")
        continue

    mask = df_dip["r:ra"].notna() & df_dip["r:dec"].notna() & df_dip["r:midpointMjdTai"].notna()
    df_clean = df_dip[mask].copy().reset_index(drop=True)

    print(f"[{field_name:12s}] computing angles for {len(df_clean):,} dipoles ...", end=" ")

    geo = compute_parallactic_zenith(
        ra_deg=df_clean["r:ra"].values,
        dec_deg=df_clean["r:dec"].values,
        mjd=df_clean["r:midpointMjdTai"].values,
    )

    df_clean = pd.concat([df_clean.reset_index(drop=True), geo.reset_index(drop=True)], axis=1)

    # ── Angular differences (using converted PA, not raw dipoleAngle) ─────────
    if "dipole_PA_deg" in df_clean.columns:
        # Signed difference modulo 360 (for scatter plots)
        df_clean["delta_PA_para"] = (df_clean["dipole_PA_deg"] - df_clean["parallactic_angle_deg"]) % 360.0
        # Folded to [0, 180] — headless axis comparison
        df_clean["delta_PA_para_folded"] = angular_difference_headless(
            df_clean["dipole_PA_deg"].values,
            df_clean["parallactic_angle_deg"].values,
        )
        # Angular difference vs azimuth (North=0, East=+90, standard astropy)
        # Comparing dipole_PA to azimuth tests whether the dipole axis tracks
        # the horizontal direction toward the target (e.g. wind-driven PSF).
        df_clean["delta_PA_az"] = (df_clean["dipole_PA_deg"] - df_clean["azimuth_deg"]) % 360.0
        df_clean["delta_PA_az_folded"] = angular_difference_headless(
            df_clean["dipole_PA_deg"].values,
            df_clean["azimuth_deg"].values,
        )

    frames_with_angles.append(df_clean)
    print("done")

if frames_with_angles:
    df_all = pd.concat(frames_with_angles, ignore_index=True)
    print(f"\nTotal dipoles with angles: {len(df_all):,}")
    print(
        df_all[
            [
                "field",
                "r:band",
                "r:dipoleAngle",
                "dipole_PA_deg",
                "parallactic_angle_deg",
                "zenith_angle_deg",
                "airmass",
            ]
        ].describe()
    )
else:
    df_all = pd.DataFrame()
    print("No dipoles found — nothing to analyse.")

### Cross-check: compare raw vs PA distributions

Both rose diagrams should look structurally identical but rotated by 90°.
This verifies that the convention conversion is applied correctly.

In [ ]:
if not df_all.empty and "dipole_PA_deg" in df_all.columns:
    n_bins = 36
    bin_edges = np.linspace(0, 360, n_bins + 1)
    bin_edges_rad = np.radians(bin_edges)
    centers_rad = (bin_edges_rad[:-1] + bin_edges_rad[1:]) / 2.0
    width_rad = 2 * np.pi / n_bins

    fig, axes = plt.subplots(1, 2, figsize=(10, 5), subplot_kw={"projection": "polar"})

    for ax, col, title in [
        (axes[0], "r:dipoleAngle", "r:dipoleAngle (raw)\nEast=0°, CCW  [same as 01d]"),
        (axes[1], "dipole_PA_deg", "dipole_PA_deg (converted)\nNorth=0°, CCW  [astro convention]"),
    ]:
        vals = df_all[col].dropna().values % 360.0
        cnts, _ = np.histogram(vals, bins=bin_edges)
        ax.bar(
            centers_rad, cnts, width=width_rad * 0.9, color="steelblue", edgecolor="white", lw=0.4, alpha=0.85
        )
        uniform = len(vals) / n_bins
        ax.plot(
            np.linspace(0, 2 * np.pi, 300),
            np.full(300, uniform),
            "--",
            color="crimson",
            lw=1.2,
            label="uniform",
        )
        ax.set_theta_zero_location("N")
        ax.set_theta_direction(-1)  # clockwise = East to the right
        ax.set_title(title, va="bottom", pad=20, fontsize=9)

    plt.suptitle("Convention cross-check: raw vs converted dipole angle", y=1.02, fontsize=11)
    plt.tight_layout()
    savefig("crosscheck_raw_vs_PA")
    plt.show()

    print("The two rose diagrams should look identical but rotated by 90°.")

## 6. Overview distributions

In [ ]:
if df_all.empty:
    print("No data — skipping.")
else:
    fig, axes = plt.subplots(1, 4, figsize=(15, 4))

    axes[0].hist(
        df_all["parallactic_angle_deg"].dropna(), bins=36, color="steelblue", edgecolor="white", lw=0.3
    )
    axes[0].set_xlabel("Parallactic angle η (deg)\n[North=0, East=+90]")
    axes[0].set_ylabel("N dipoles")
    axes[0].set_title("Parallactic angle")

    axes[1].hist(df_all["zenith_angle_deg"].dropna(), bins=30, color="darkorange", edgecolor="white", lw=0.3)
    axes[1].set_xlabel("Zenith angle z (deg)")
    axes[1].set_title("Zenith angle")

    axes[2].hist(df_all["airmass"].dropna().clip(1, 3), bins=30, color="seagreen", edgecolor="white", lw=0.3)
    axes[2].set_xlabel("Airmass X")
    axes[2].set_title("Airmass")

    axes[3].hist(df_all["dipole_PA_deg"].dropna(), bins=36, color="crimson", edgecolor="white", lw=0.3)
    axes[3].set_xlabel("Dipole PA (deg)\n[North=0, East=+90]")
    axes[3].set_title("Dipole PA (converted)")

    fig2, axes2 = plt.subplots(1, 1, figsize=(5, 4))
    axes2.hist(df_all["azimuth_deg"].dropna(), bins=36, color="mediumpurple", edgecolor="white", lw=0.3)
    axes2.set_xlabel("Azimuth (deg)\n[North=0, East=+90]")
    axes2.set_ylabel("N dipoles")
    axes2.set_title("Azimuth")
    plt.tight_layout()
    savefig("overview_azimuth_distribution")
    plt.show()

    plt.suptitle("All DDFs combined — overview", y=1.02, fontsize=11)
    plt.tight_layout()
    savefig("overview_distributions")
    plt.show()

## 7. Dipole PA vs parallactic angle — scatter & 2D histogram (per DDF)

Both quantities are now in the same astronomical convention (North = 0°, CCW).

In [ ]:
def plot_dipole_vs_angle(
    df: pd.DataFrame,
    x_col: str,
    x_label: str,
    y_col: str,
    y_label: str,
    field_name: str,
    figname_suffix: str,
) -> None:
    """
    Two-panel figure:
      Left  — scatter (x_col vs y_col) coloured by band
      Right — 2D histogram density
    Prints Pearson and Spearman correlation coefficients.
    """
    mask = df[x_col].notna() & df[y_col].notna()
    sub = df[mask].copy()
    if len(sub) < 5:
        print(f"  [{field_name}] too few points — skipping.")
        return

    x = sub[x_col].values
    y = sub[y_col].values

    r_p, p_p = stats.pearsonr(x, y)
    r_s, p_s = stats.spearmanr(x, y)
    stat_str = f"Pearson r={r_p:.3f} (p={p_p:.2e})   Spearman ρ={r_s:.3f} (p={p_s:.2e})"
    print(f"  [{field_name}] {stat_str}")

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    if "r:band" in sub.columns:
        for band in [b for b in BAND_ORDER if b in sub["r:band"].dropna().unique()]:
            idx = sub["r:band"] == band
            ax1.scatter(
                sub.loc[idx, x_col],
                sub.loc[idx, y_col],
                s=4,
                alpha=0.5,
                color=BAND_COLORS.get(band, "grey"),
                label=f"{band} (n={idx.sum():,})",
                rasterized=True,
            )
        ax1.legend(fontsize=7, markerscale=2, loc="best")
    else:
        ax1.scatter(x, y, s=4, alpha=0.4, color="steelblue", rasterized=True)

    ax1.set_xlabel(x_label)
    ax1.set_ylabel(y_label)
    ax1.set_title(f"{field_name} — scatter\n{stat_str}", fontsize=8)

    h2 = ax2.hist2d(x, y, bins=[40, 36], cmap="viridis", norm=mcolors.LogNorm(vmin=1))
    plt.colorbar(h2[3], ax=ax2, label="N dipoles (log scale)")
    ax2.set_xlabel(x_label)
    ax2.set_ylabel(y_label)
    ax2.set_title(f"{field_name} — 2D histogram")

    plt.tight_layout()
    savefig(f"{field_name.replace('-', '_')}_{figname_suffix}")
    plt.show()


print("Helper function defined.")

In [ ]:
print("=" * 70)
print("Dipole PA vs PARALLACTIC angle — per DDF")
print("(both in North=0°, CCW astronomical convention)")
print("=" * 70)

for field_name in DEEP_FIELDS:
    sub = df_all[df_all["field"] == field_name] if not df_all.empty else pd.DataFrame()
    if sub.empty:
        continue
    plot_dipole_vs_angle(
        df=sub,
        x_col="parallactic_angle_deg",
        x_label="Parallactic angle η (deg)  [North=0, East=+90]",
        y_col="dipole_PA_deg",
        y_label="Dipole PA (deg)  [North=0, East=+90]",
        field_name=field_name,
        figname_suffix="dipolePA_vs_parallactic",
    )

In [ ]:
print("=" * 70)
print("Dipole PA vs ZENITH angle — per DDF")
print("=" * 70)

for field_name in DEEP_FIELDS:
    sub = df_all[df_all["field"] == field_name] if not df_all.empty else pd.DataFrame()
    if sub.empty:
        continue
    plot_dipole_vs_angle(
        df=sub,
        x_col="zenith_angle_deg",
        x_label="Zenith angle z (deg)",
        y_col="dipole_PA_deg",
        y_label="Dipole PA (deg)  [North=0, East=+90]",
        field_name=field_name,
        figname_suffix="dipolePA_vs_zenith",
    )

In [ ]:
print("=" * 70)
print("Dipole PA vs AZIMUTH — per DDF")
print("(azimuth: North=0°, East=+90° — standard astropy convention)")
print("=" * 70)

for field_name in DEEP_FIELDS:
    sub = df_all[df_all["field"] == field_name] if not df_all.empty else pd.DataFrame()
    if sub.empty:
        continue
    plot_dipole_vs_angle(
        df=sub,
        x_col="azimuth_deg",
        x_label="Azimuth (deg)  [North=0, East=+90]",
        y_col="dipole_PA_deg",
        y_label="Dipole PA (deg)  [North=0, East=+90]",
        field_name=field_name,
        figname_suffix="dipolePA_vs_azimuth",
    )

## 8. Dipole length vs parallactic angle, zenith angle, and airmass

In [ ]:
for field_name in DEEP_FIELDS:
    sub = df_all[df_all["field"] == field_name] if not df_all.empty else pd.DataFrame()
    if sub.empty:
        continue
    for x_col, x_label, suffix in [
        ("parallactic_angle_deg", "Parallactic angle η (deg)", "dipoleLength_vs_parallactic"),
        ("zenith_angle_deg", "Zenith angle z (deg)", "dipoleLength_vs_zenith"),
        ("airmass", "Airmass X", "dipoleLength_vs_airmass"),
        ("azimuth_deg", "Azimuth (deg)  [North=0, East=+90]", "dipoleLength_vs_azimuth"),
    ]:
        plot_dipole_vs_angle(
            df=sub,
            x_col=x_col,
            x_label=x_label,
            y_col="r:dipoleLength",
            y_label="Dipole length (arcsec)",
            field_name=field_name,
            figname_suffix=suffix,
        )

## 9. Angular difference Δ = dipole_PA − parallactic_angle

Both angles are now in the **same** astronomical convention (North = 0°, CCW).

**Physical interpretation:**
* If dipoles are caused by atmospheric dispersion / differential refraction, they should point
  along the great circle toward the zenith, i.e. **aligned with the parallactic angle**.
* A dipole axis is headless (0° ≡ 180°), so we fold Δ to [0°, 180°].
* A concentration near **0° or 180°** → axis aligned with the zenith direction.
* A concentration near **90°** → axis perpendicular to the zenith direction.
* A **uniform distribution** → no preferred orientation relative to the zenith.

In [ ]:
def polar_rose_delta(
    df: pd.DataFrame,
    angle_col: str,
    title: str,
    figname: str,
    n_bins: int = 36,
    angle_range: tuple = (0, 360),
) -> None:
    """Polar histogram of *angle_col* (in degrees), stacked by band."""
    if df.empty or angle_col not in df.columns:
        return

    lo, hi = angle_range
    bin_edges = np.linspace(lo, hi, n_bins + 1)
    bin_edges_rad = np.radians(bin_edges)
    centers_rad = (bin_edges_rad[:-1] + bin_edges_rad[1:]) / 2.0
    width_rad = np.radians(hi - lo) / n_bins

    fig = plt.figure(figsize=(5, 5))
    theta_max = np.radians(hi)
    ax = fig.add_subplot(111, projection="polar")

    bottom = np.zeros(n_bins)
    n_total = 0

    if "r:band" in df.columns:
        for band in [b for b in BAND_ORDER if b in df["r:band"].dropna().unique()]:
            vals = df.loc[df["r:band"] == band, angle_col].dropna().values
            vals = vals[(vals >= lo) & (vals <= hi)]
            if len(vals) == 0:
                continue
            n_total += len(vals)
            cnts, _ = np.histogram(vals, bins=bin_edges)
            ax.bar(
                centers_rad,
                cnts,
                width=width_rad * 0.9,
                bottom=bottom,
                color=BAND_COLORS.get(band, "grey"),
                edgecolor="white",
                linewidth=0.4,
                alpha=0.85,
                label=f"{band} (n={len(vals):,})",
            )
            bottom += cnts
        ax.legend(loc="lower right", fontsize=7, bbox_to_anchor=(1.30, -0.05))
    else:
        vals = df[angle_col].dropna().values
        vals = vals[(vals >= lo) & (vals <= hi)]
        n_total = len(vals)
        cnts, _ = np.histogram(vals, bins=bin_edges)
        ax.bar(centers_rad, cnts, width=width_rad * 0.9, color="steelblue", edgecolor="white", linewidth=0.4)

    if n_total > 0:
        uniform = n_total / n_bins
        ax.plot(
            np.linspace(0, theta_max, 300),
            np.full(300, uniform),
            "--",
            color="crimson",
            lw=1.2,
            alpha=0.85,
            label="uniform",
        )

    ax.set_theta_zero_location("N")
    ax.set_theta_direction(-1)
    ax.set_thetalim(0, theta_max)
    ax.set_title(f"{title}\n(n={n_total:,})", va="bottom", pad=20)
    plt.tight_layout()
    savefig(figname)
    plt.show()


print("polar_rose_delta helper defined.")

In [ ]:
# Global folded Δ  (0–180° half-circle since dipole axis is headless)
polar_rose_delta(
    df_all,
    "delta_PA_para_folded",
    title="|dipole_PA − parallactic_angle|  [folded 0–180°]\nAll DDFs",
    figname="all_delta_PA_para_folded_polar",
    n_bins=18,
    angle_range=(0, 180),
)

# Full 360° rose of signed Δ
polar_rose_delta(
    df_all,
    "delta_PA_para",
    title="dipole_PA − parallactic_angle  [0–360°]\nAll DDFs",
    figname="all_delta_PA_para_360_polar",
    n_bins=36,
    angle_range=(0, 360),
)

In [ ]:
# Per-DDF rose diagrams
for field_name in DEEP_FIELDS:
    sub = df_all[df_all["field"] == field_name] if not df_all.empty else pd.DataFrame()
    if sub.empty or "delta_PA_para_folded" not in sub.columns:
        continue
    polar_rose_delta(
        sub,
        "delta_PA_para_folded",
        title=f"{field_name} — |dipole_PA − η|  [folded 0–180°]",
        figname=f"{field_name.replace('-', '_')}_delta_PA_para_folded_polar",
        n_bins=18,
        angle_range=(0, 180),
    )

In [ ]:
# ── Azimuth-based rose diagrams ──────────────────────────────────────────────
# delta_PA_az_folded: |dipole_PA - azimuth| folded to [0, 180]
# Physical interpretation:
#   peaked near 0°  → dipole axis aligned with the azimuth (horizontal direction toward target)
#   peaked near 90° → dipole axis perpendicular to azimuth direction
#   uniform         → no preferred orientation w.r.t. azimuth

# Global folded Δ vs azimuth
polar_rose_delta(
    df_all,
    "delta_PA_az_folded",
    title="|dipole_PA − azimuth|  [folded 0–180°]\nAll DDFs",
    figname="all_delta_PA_az_folded_polar",
    n_bins=18,
    angle_range=(0, 180),
)

# Full 360° rose of signed Δ vs azimuth
polar_rose_delta(
    df_all,
    "delta_PA_az",
    title="dipole_PA − azimuth  [0–360°]\nAll DDFs",
    figname="all_delta_PA_az_360_polar",
    n_bins=36,
    angle_range=(0, 360),
)

In [ ]:
# Per-DDF rose diagrams vs azimuth
for field_name in DEEP_FIELDS:
    sub = df_all[df_all["field"] == field_name] if not df_all.empty else pd.DataFrame()
    if sub.empty or "delta_PA_az_folded" not in sub.columns:
        continue
    polar_rose_delta(
        sub,
        "delta_PA_az_folded",
        title=f"{field_name} — |dipole_PA − azimuth|  [folded 0–180°]",
        figname=f"{field_name.replace('-', '_')}_delta_PA_az_folded_polar",
        n_bins=18,
        angle_range=(0, 180),
    )

### 9b. Azimuth rose diagrams — subplots by band, per DDF

Each panel below shows a **polar histogram of `delta_PA_az_folded`** (= |dipole_PA − azimuth|
folded to [0°, 180°]) for a **single photometric band**, colour-stacked by DDF field.
Two figures are produced:

1. **All DDFs combined** — one polar subplot per band (6 panels).
2. **Per DDF** — one figure per field, one polar subplot per band.

In [ ]:
# ── Rose subplots by band — all DDFs combined ────────────────────────────────
# One polar subplot per band; bars coloured by DDF field.
# Angle variable: delta_PA_az_folded in [0, 180] (headless axis comparison).


def rose_subplots_by_band_all_ddfs(
    df: pd.DataFrame,
    angle_col: str = "delta_PA_az_folded",
    angle_range: tuple = (0, 180),
    n_bins: int = 18,
    figname: str = "all_ddfs_azrose_by_band",
) -> None:
    """
    Plot a grid of polar-rose subplots, one per photometric band,
    for all DDFs combined.  Bars are colour-coded by DDF field.

    Parameters
    ----------
    df : pd.DataFrame — concatenated alert table with columns
        ``r:band``, ``field``, and ``angle_col``.
    angle_col : str — column of angle values in degrees.
    angle_range : (lo, hi) in degrees.
    n_bins : int — number of angular bins.
    figname : str — base name for saved files.
    """
    if df.empty or angle_col not in df.columns or "r:band" not in df.columns:
        print(f"[rose_subplots_by_band_all_ddfs] missing data — skipping.")
        return

    bands_present = [b for b in BAND_ORDER if b in df["r:band"].dropna().unique()]
    if not bands_present:
        return

    fields_present = [f for f in DEEP_FIELDS if f in df["field"].dropna().unique()]
    field_colors = {f: plt.get_cmap("tab10", len(fields_present))(k) for k, f in enumerate(fields_present)}

    lo, hi = angle_range
    bin_edges = np.linspace(lo, hi, n_bins + 1)
    bin_edges_rad = np.radians(bin_edges)
    centers_rad = (bin_edges_rad[:-1] + bin_edges_rad[1:]) / 2.0
    width_rad = np.radians(hi - lo) / n_bins
    theta_max = np.radians(hi)

    ncols = min(3, len(bands_present))
    nrows = int(np.ceil(len(bands_present) / ncols))
    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(ncols * 4, nrows * 4),
        subplot_kw={"projection": "polar"},
        squeeze=False,
    )

    for bidx, band in enumerate(bands_present):
        ax = axes[bidx // ncols][bidx % ncols]
        sub_band = df[df["r:band"] == band]

        bottom = np.zeros(n_bins)
        n_total = 0

        for field_name in fields_present:
            vals = sub_band.loc[sub_band["field"] == field_name, angle_col].dropna().values
            vals = vals[(vals >= lo) & (vals <= hi)]
            if len(vals) == 0:
                continue
            n_total += len(vals)
            cnts, _ = np.histogram(vals, bins=bin_edges)
            ax.bar(
                centers_rad,
                cnts,
                width=width_rad * 0.9,
                bottom=bottom,
                color=field_colors[field_name],
                edgecolor="white",
                linewidth=0.3,
                alpha=0.85,
                label=field_name,
            )
            bottom += cnts

        # Uniform reference line
        if n_total > 0:
            uniform = n_total / n_bins
            ax.plot(
                np.linspace(0, theta_max, 300),
                np.full(300, uniform),
                "--",
                color="crimson",
                lw=1.0,
                alpha=0.8,
            )

        ax.set_theta_zero_location("N")
        ax.set_theta_direction(-1)
        ax.set_thetalim(0, theta_max)
        ax.set_title(
            f"Band {band}  (n={n_total:,})",
            va="bottom",
            pad=14,
            fontsize=9,
        )
        # Legend only on first subplot to save space
        if bidx == 0:
            ax.legend(
                loc="lower right",
                fontsize=6,
                bbox_to_anchor=(1.55, -0.05),
            )

    # Hide unused subplots
    for bidx in range(len(bands_present), nrows * ncols):
        axes[bidx // ncols][bidx % ncols].set_visible(False)

    fig.suptitle(
        f"|dipole_PA − azimuth|  [folded 0–180°]\nRose diagrams by band — all DDFs combined",
        y=1.02,
        fontsize=11,
    )
    plt.tight_layout()
    savefig(figname)
    plt.show()


rose_subplots_by_band_all_ddfs(
    df_all,
    angle_col="delta_PA_az_folded",
    angle_range=(0, 180),
    n_bins=18,
    figname="all_ddfs_azrose_by_band",
)

In [ ]:
# ── Rose subplots by band — one figure per DDF ───────────────────────────────
# For each DDF: one polar subplot per band, bars coloured by band.


def rose_subplots_by_band_per_ddf(
    df: pd.DataFrame,
    field_name: str,
    angle_col: str = "delta_PA_az_folded",
    angle_range: tuple = (0, 180),
    n_bins: int = 18,
    figname_prefix: str = "azrose_by_band",
) -> None:
    """
    For a single DDF ``field_name``, plot a grid of polar-rose subplots,
    one per photometric band, using ``BAND_COLORS``.

    Parameters
    ----------
    df : pd.DataFrame — alert table filtered (or not) to one field.
    field_name : str — DDF label used in titles and filenames.
    angle_col : str — column of angle values in degrees.
    angle_range : (lo, hi) in degrees.
    n_bins : int — number of angular bins.
    figname_prefix : str — prefix for saved files.
    """
    if df.empty or angle_col not in df.columns or "r:band" not in df.columns:
        return

    bands_present = [b for b in BAND_ORDER if b in df["r:band"].dropna().unique()]
    if not bands_present:
        return

    lo, hi = angle_range
    bin_edges = np.linspace(lo, hi, n_bins + 1)
    bin_edges_rad = np.radians(bin_edges)
    centers_rad = (bin_edges_rad[:-1] + bin_edges_rad[1:]) / 2.0
    width_rad = np.radians(hi - lo) / n_bins
    theta_max = np.radians(hi)

    ncols = min(3, len(bands_present))
    nrows = int(np.ceil(len(bands_present) / ncols))
    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(ncols * 4, nrows * 4),
        subplot_kw={"projection": "polar"},
        squeeze=False,
    )

    for bidx, band in enumerate(bands_present):
        ax = axes[bidx // ncols][bidx % ncols]
        vals = df.loc[df["r:band"] == band, angle_col].dropna().values
        vals = vals[(vals >= lo) & (vals <= hi)]
        n_total = len(vals)

        if n_total > 0:
            cnts, _ = np.histogram(vals, bins=bin_edges)
            ax.bar(
                centers_rad,
                cnts,
                width=width_rad * 0.9,
                color=BAND_COLORS.get(band, "grey"),
                edgecolor="white",
                linewidth=0.3,
                alpha=0.85,
            )
            # Uniform reference line
            uniform = n_total / n_bins
            ax.plot(
                np.linspace(0, theta_max, 300),
                np.full(300, uniform),
                "--",
                color="crimson",
                lw=1.0,
                alpha=0.8,
                label="uniform",
            )
            ax.legend(loc="lower right", fontsize=6, bbox_to_anchor=(1.3, -0.05))

        ax.set_theta_zero_location("N")
        ax.set_theta_direction(-1)
        ax.set_thetalim(0, theta_max)
        ax.set_title(
            f"Band {band}  (n={n_total:,})",
            va="bottom",
            pad=14,
            fontsize=9,
        )

    # Hide unused subplots
    for bidx in range(len(bands_present), nrows * ncols):
        axes[bidx // ncols][bidx % ncols].set_visible(False)

    fig.suptitle(
        f"{field_name} — |dipole_PA − azimuth|  [folded 0–180°]\nRose diagrams by band",
        y=1.02,
        fontsize=11,
    )
    plt.tight_layout()
    savefig(f"{field_name.replace('-', '_')}_{figname_prefix}")
    plt.show()


for field_name in DEEP_FIELDS:
    sub = df_all[df_all["field"] == field_name] if not df_all.empty else pd.DataFrame()
    if sub.empty or "delta_PA_az_folded" not in sub.columns:
        continue
    rose_subplots_by_band_per_ddf(
        sub,
        field_name=field_name,
        angle_col="delta_PA_az_folded",
        angle_range=(0, 180),
        n_bins=18,
        figname_prefix="azrose_by_band",
    )

## 10. Band-resolved correlation summary

In [ ]:
rows = []

for field_name in DEEP_FIELDS:
    sub_field = df_all[df_all["field"] == field_name] if not df_all.empty else pd.DataFrame()
    if sub_field.empty:
        continue

    bands_present = sub_field["r:band"].dropna().unique() if "r:band" in sub_field.columns else ["all"]
    for band in bands_present:
        sub = sub_field[sub_field["r:band"] == band] if band != "all" else sub_field

        for x_col, x_label in [
            ("parallactic_angle_deg", "parallactic"),
            ("zenith_angle_deg", "zenith"),
            ("airmass", "airmass"),
            ("azimuth_deg", "azimuth"),
        ]:
            for y_col, y_label in [
                ("dipole_PA_deg", "dipole_PA"),
                ("r:dipoleLength", "dipoleLength"),
                ("delta_PA_para_folded", "delta_PA_folded"),
                ("delta_PA_az_folded", "delta_PA_az_folded"),
            ]:
                if y_col not in sub.columns:
                    continue
                mask = sub[x_col].notna() & sub[y_col].notna()
                x = sub.loc[mask, x_col].values
                y = sub.loc[mask, y_col].values
                if len(x) < 5:
                    continue
                r_p, p_p = stats.pearsonr(x, y)
                r_s, p_s = stats.spearmanr(x, y)
                rows.append(
                    {
                        "field": field_name,
                        "band": band,
                        "x": x_label,
                        "y": y_label,
                        "n": int(mask.sum()),
                        "pearson_r": round(r_p, 4),
                        "pearson_p": round(p_p, 4),
                        "spearman_r": round(r_s, 4),
                        "spearman_p": round(p_s, 4),
                    }
                )

df_corr = pd.DataFrame(rows)
if not df_corr.empty:
    pd.set_option("display.max_rows", 120)
    display(df_corr.sort_values(["field", "band", "x", "y"]))
else:
    print("No correlation data computed.")

In [ ]:
# Spearman ρ heatmaps
if not df_corr.empty:
    for y_label, x_label, title_suffix in [
        ("dipole_PA", "parallactic", "DipolePA vs Parallactic"),
        ("dipole_PA", "zenith", "DipolePA vs Zenith"),
        ("dipoleLength", "airmass", "DipoleLength vs Airmass"),
        ("delta_PA_folded", "parallactic", "delta_PA_folded vs Parallactic"),
        ("delta_PA_folded", "zenith", "delta_PA_folded vs Zenith"),
        ("dipole_PA", "azimuth", "DipolePA vs Azimuth"),
        ("dipoleLength", "azimuth", "DipoleLength vs Azimuth"),
        ("delta_PA_az_folded", "azimuth", "delta_PA_az_folded vs Azimuth"),
    ]:
        sub_heat = df_corr[(df_corr["x"] == x_label) & (df_corr["y"] == y_label)]
        if sub_heat.empty:
            continue

        pivot = sub_heat.pivot_table(index="field", columns="band", values="spearman_r").reindex(
            columns=BAND_ORDER
        )

        fig, ax = plt.subplots(figsize=(8, max(3, len(pivot) * 0.6)))
        im = ax.imshow(pivot.values, aspect="auto", cmap="RdBu_r", vmin=-1, vmax=1)
        plt.colorbar(im, ax=ax, label="Spearman ρ")
        ax.set_xticks(range(pivot.shape[1]))
        ax.set_xticklabels(pivot.columns.tolist())
        ax.set_yticks(range(pivot.shape[0]))
        ax.set_yticklabels(pivot.index.tolist())
        ax.set_xlabel("Band")
        ax.set_ylabel("DDF")
        ax.set_title(f"Spearman ρ — {title_suffix}")

        for i in range(pivot.shape[0]):
            for j in range(pivot.shape[1]):
                val = pivot.values[i, j]
                if not np.isnan(val):
                    ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=8, color="black")

        plt.tight_layout()
        savefig(f"heatmap_spearman_{title_suffix.lower().replace(' ', '_')}")
        plt.show()

## 11. Per-band panels — dipole PA vs η and vs z (all DDFs combined)

In [ ]:
if df_all.empty or "r:band" not in df_all.columns:
    print("No data — skipping per-band plots.")
else:
    bands_present = [b for b in BAND_ORDER if b in df_all["r:band"].dropna().unique()]

    for x_col, x_label, figname_base in [
        (
            "parallactic_angle_deg",
            "Parallactic angle η (deg)  [North=0, E=+90]",
            "all_ddfs_dipolePA_vs_parallactic_by_band",
        ),
        ("zenith_angle_deg", "Zenith angle z (deg)", "all_ddfs_dipolePA_vs_zenith_by_band"),
        ("azimuth_deg", "Azimuth (deg)  [North=0, E=+90]", "all_ddfs_dipolePA_vs_azimuth_by_band"),
    ]:
        ncols = min(3, len(bands_present))
        nrows = int(np.ceil(len(bands_present) / ncols))
        fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 5, nrows * 4), squeeze=False)

        for idx, band in enumerate(bands_present):
            ax = axes[idx // ncols][idx % ncols]
            sub = df_all[(df_all["r:band"] == band) & df_all[x_col].notna() & df_all["dipole_PA_deg"].notna()]
            if len(sub) < 5:
                ax.set_visible(False)
                continue

            x = sub[x_col].values
            y = sub["dipole_PA_deg"].values
            r_s, p_s = stats.spearmanr(x, y)

            cmap_f = plt.get_cmap("tab10", len(DEEP_FIELDS))
            for k, fld in enumerate(sorted(sub["field"].dropna().unique())):
                idx_f = sub["field"] == fld
                ax.scatter(
                    sub.loc[idx_f, x_col],
                    sub.loc[idx_f, "dipole_PA_deg"],
                    s=4,
                    alpha=0.4,
                    color=cmap_f(k),
                    label=fld,
                    rasterized=True,
                )

            ax.set_xlabel(x_label, fontsize=8)
            ax.set_ylabel("Dipole PA (deg)  [North=0, East=+90]", fontsize=8)
            ax.set_title(
                f"Band {band}  (n={len(sub):,})\nSpearman ρ={r_s:.3f}  p={p_s:.2e}",
                fontsize=8,
            )
            ax.legend(fontsize=6, markerscale=2, loc="best")

        for idx in range(len(bands_present), nrows * ncols):
            axes[idx // ncols][idx % ncols].set_visible(False)

        plt.suptitle(
            f"Dipole PA vs {x_label} — all DDFs, per band\n"
            "(dipole_PA_deg in astronomical convention: North=0°, East=+90°)",
            y=1.01,
            fontsize=10,
        )
        plt.tight_layout()
        savefig(figname_base)
        plt.show()

## 12. Summary

### Angle conventions used in this notebook

| Column | Convention | Origin |
|--------|------------|--------|
| `r:dipoleAngle` | East = 0°, CCW (pixel/image) | Rubin AP pipeline |
| `dipole_PA_deg` | **North = 0°, CCW** (astronomical PA) | = (90 − `r:dipoleAngle`) mod 360 |
| `parallactic_angle_deg` | **North = 0°, CCW** | astropy `AltAz.parallactic_angle()` |
| `azimuth_deg` | **North = 0°, East = +90°** | astropy `AltAz.az` |
| `delta_PA_para` | (dipole_PA − η) mod 360 | signed difference |
| `delta_PA_para_folded` | \|dipole_PA − η\| folded to [0°, 180°] | headless-axis comparison |
| `delta_PA_az` | (dipole_PA − az) mod 360 | signed difference vs azimuth |
| `delta_PA_az_folded` | \|dipole_PA − az\| folded to [0°, 180°] | headless-axis comparison vs azimuth |

### Physical interpretation

| Result | Interpretation |
|--------|----------------|
| `delta_PA_para_folded` peaked near **0°** | Dipoles aligned with zenith direction → atmospheric origin |
| `delta_PA_para_folded` peaked near **90°** | Dipoles perpendicular to zenith → other origin |
| `delta_PA_para_folded` **uniform** | No preferred orientation relative to zenith |
| `dipoleLength` grows with **airmass** | Atmospheric dispersion stretching the PSF |
| `delta_PA_az_folded` peaked near **0°** | Dipoles aligned with azimuth direction → wind/dome seeing? |
| `delta_PA_az_folded` peaked near **90°** | Dipoles perpendicular to azimuth |
| `delta_PA_az_folded` **uniform** | No preferred orientation relative to azimuth |

All figures saved to `figs_DIPOLES_05/`.
